# Task 5: Positional Embeddings

Learn two types: Sinusoidal (fixed) and Learned embeddings.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

## Sinusoidal Positional Embeddings

Using sine and cosine functions at different frequencies.

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding."""
    
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

# Test
pe = PositionalEncoding(d_model=128, max_len=100)
x = torch.randn(1, 50, 128)
out = pe(x)
print(f"Input: {x.shape} -> Output: {out.shape}")

In [ ]:
# Visualize positional encodings
pe_layer = PositionalEncoding(d_model=64, max_len=100)
pos_embeddings = pe_layer.pe[0].T  # (d_model, max_len)

plt.figure(figsize=(12, 5))
plt.imshow(pos_embeddings, cmap='RdBu', aspect='auto')
plt.colorbar()
plt.xlabel('Position')
plt.ylabel('Embedding Dimension')
plt.title('Sinusoidal Positional Embeddings')
plt.show()

## Learned Positional Embeddings

BERT uses learned positional embeddings instead.

In [ ]:
class LearnedPositionalEmbedding(nn.Module):
    """Learned positional embeddings (used in BERT)."""
    
    def __init__(self, d_model, max_len=512):
        super().__init__()
        self.position_embeddings = nn.Embedding(max_len, d_model)
        
    def forward(self, x):
        batch_size, seq_len = x.shape[:2]
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, -1)
        return x + self.position_embeddings(positions)

# Test
lpe = LearnedPositionalEmbedding(d_model=128)
x = torch.randn(2, 50, 128)
out = lpe(x)
print(f"Input: {x.shape} -> Output: {out.shape}")

## Summary
- ✅ Sinusoidal: Fixed, extrapolates to longer sequences
- ✅ Learned: More flexible, BERT uses this

## Next: [Task 6: BERT Embeddings & Encoder](../task-06-bert-embeddings-encoder/overview.html)